# 07 · Baselines (task 5.5)

Rule-based forecasts that any model has to beat, scored on the latest stretch of data held out by time (the first 70% trains, the rest tests; with weeks of data, the latest 2 weeks). Nothing after the split leaks into training.

**Lifespan** (the days horizon, here per list): at every hourly fetch, for every item in a list's top 10, how many more hours it stays. Rules:
- `median_left`: the Kaplan–Meier median of the time left at the item's age, fitted on training stays;
- `as_long_again`: it lasts as long again as it has so far, at least an hour (the Lindy rule);
- `one_more_hour`.

The fair score is **"still listed k hours later?"**. It's scored on every forecast made at least k hours before the end of the data, whose answer is known whether the stay ended or not. `still_listed` is the true share, so a rule must beat always giving the more common answer. Error in hours is shown too, but only stays that ended can be scored that way, which favors short stays when the test stretch is short.

**Breakout** (the hours horizon): PLAN.md's rule, "on 2 platforms within 2 hours of first appearing", forecasts a story reaching 3 or more platforms within 6 hours. Scored by precision, recall and lead time, on hours when every list was collected and the next 6 hours are in the data.

**Needs** `npm ci` at the repo root for the breakout part (as notebook 06 does). Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import pandas as pd

from topnews import baselines, db, echo, leadlag, lifecycle, rhythms

engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # YouTube left out
fast = items[~items.source_id.isin(rhythms.SLOW_SOURCES)]
split, end = baselines.split_time(fast), fast.fetched_at.max()
print(f"train {fast.fetched_at.min():%m-%d %H:%M} to {split:%m-%d %H:%M} UTC, test to {end:%m-%d %H:%M}")

## Lifespan: still listed k hours later?

Strict, and with one missed fetch allowed (Bluesky flickers, RQ1).

In [ ]:
lifespan = {}
for name, missed in [("strict", 0), ("one missed fetch allowed", 1)]:
    spells = lifecycle.spells(fast, max_missed=missed)
    table = baselines.median_left_table(baselines.training_spells(spells, split))
    lifespan[name] = baselines.lifespan_forecasts(spells, split, table)
    print(name)
    display(baselines.score_horizons(lifespan[name], end).pivot_table(
        index=["list_id", "k_hours", "forecasts", "still_listed"], columns="rule", values="accuracy").round(2))

## Lifespan: error in hours, on stays that ended

Biased toward short stays while the test stretch is short; read with the table above.

In [ ]:
baselines.score_lifespans(lifespan["strict"]).round(2)

## Breakout: on 2 platforms within 2 hours, then 3 within 6?

In [ ]:
since = echo.since_all_present(items)
cp, _ = echo.analyze(since, name="baselines")
stories = leadlag.story_pairs(leadlag.lead_pairs(cp, leadlag.source_starts(since)))
starts = baselines.story_platform_starts(stories) if len(stories) else pd.DataFrame(columns=["story", "platform", "first", "censored"])
hours = sorted(since.fetched_at.dt.floor("h").unique())
forecasts = baselines.breakout_forecasts(starts, hours, since.fetched_at.max())
print(f"{starts.story.nunique()} matched stories since every list was collected; {len(forecasts)} story-hours with 6 hours of data after them")
baselines.score_breakouts(forecasts)